<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/03-dataframe-fundamentals/03-sort-distinct-limit-sample.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 3.3 — orderBy/sort, distinct, limit, sample

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("3.3-sort-distinct-limit-sample")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## orderBy — and where nulls land

In [ ]:
# Ascending: the null-quantity rows (1012, 1032) come FIRST
orders.orderBy("quantity").select("order_id", "quantity", "product").show(5)

In [ ]:
# Fix: explicit null placement + multi-key sort
orders.orderBy(
    col("quantity").asc_nulls_last(),
    col("unit_price").desc(),
).select("order_id", "quantity", "unit_price").show(5)

## distinct / dropDuplicates — order 1021 is duplicated on purpose

In [ ]:
print("rows:          ", orders.count())
print("distinct rows: ", orders.distinct().count())
orders.select("country").distinct().show()

In [ ]:
# dropDuplicates by key: c001 has 3 orders — which one survives? (Arbitrary!)
orders.filter(col("customer_id") == "c001").show()
orders.dropDuplicates(["customer_id"]).filter(col("customer_id") == "c001").show()

## limit — and the orderBy+limit top-N pattern

In [ ]:
# Top 5 most expensive products — the canonical top-N
top5 = orders.orderBy(col("unit_price").desc()).limit(5)
top5.select("product", "unit_price").show()

# Module 11 preview — TakeOrderedAndProject in the plan means 'no full sort ran':
top5.explain()

## sample — approximate, seeded, honest

In [ ]:
# fraction=0.2 of 41 rows is 'about 8' — run this cell and see the spread:
for seed in (1, 2, 3, 4, 5):
    print(f"seed {seed}: {orders.sample(fraction=0.2, seed=seed).count()} rows")

# Same seed = same rows, every time (reproducibility):
print(orders.sample(fraction=0.2, seed=42).count(),
      "==", orders.sample(fraction=0.2, seed=42).count())

In [ ]:
# Stratified: keep half of IN orders, 10% of the rest
fractions = {"IN": 0.5, "US": 0.1, "DE": 0.1, "UK": 0.1}
orders.na.drop(subset=["country"]).sampleBy("country", fractions, seed=7) \
    .groupBy("country").count().show()

## Exercises

1. Top 3 orders by `line_total` (compute it first) per the top-N pattern. Then check `.explain()` — full sort or TakeOrderedAndProject?
2. Deduplicate `orders` on (`customer_id`, `order_date`) and explain in one sentence when this is safe vs when you'd need Module 8's window approach.
3. Run `orders.limit(3).show()` five times. Same rows each time? Now add `.orderBy("order_id")` before the limit — and explain why 'first 3 rows' was meaningless before.
4. How many rows does `sample(fraction=1.0, seed=1)` return — exactly 41 or approximately 41? Reason first, then run.

In [ ]:
# your exercise code here
